# Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
K, Gama, lam = 1.0, 1.0, 1.0     # uteži energije (kot na vaji 05)
R, n = 1.0, 6                    # polmer in število oglišč celice

# Funkcije

In [ ]:
# --- Model celice z vaje 05 ---
def poligon(R, n, eps=0.0, rng=rng):
    dfi = 2 * np.pi / n
    fi = dfi * np.arange(n) + rng.uniform(-eps, eps, n) * dfi
    r = R * (1 + rng.uniform(-eps, eps, n))
    return np.column_stack([r * np.cos(fi), r * np.sin(fi)])


def stranice(t):
    return np.linalg.norm(np.roll(t, -1, axis=0) - t, axis=1)


def obseg(t):
    return stranice(t).sum()


def ploscina(t):
    x, y = t[:, 0], t[:, 1]
    return 0.5 * abs(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y))


def tezisce(t):
    return t.mean(axis=0)


def referencne(R, n):
    l0 = 2 * R * np.sin(np.pi / n)
    return 0.5 * n * R**2 * np.sin(2 * np.pi / n), n * l0, l0


def energija(t, ref):
    A0, P0, l0 = ref
    return (0.5 * K * (ploscina(t) - A0)**2 + 0.5 * Gama * (obseg(t) - P0)**2
            + 0.5 * lam * np.sum((stranice(t) - l0)**2))


def narisi_celico(ax, t, barva="tab:orange", **kw):
    z = np.vstack([t, t[:1]])
    ax.fill(z[:, 0], z[:, 1], color=barva, alpha=0.3)
    ax.plot(z[:, 0], z[:, 1], "o-", color=barva, ms=3, **kw)
    ax.set_aspect("equal")


# --- Gradientni spust ---
def gs_1d(x, k, eta, n_iter=50, x0=1.0):
    """Gradientni spust za E(x) = k (x - x0)^2. Vrne zaporedje x."""
    pot = [x]
    for _ in range(n_iter):
        # TODO: x <- x - eta * dE/dx
        pass
        pot.append(x)
    return np.array(pot)


def gradient(t, ref, h=1e-6):
    """Numerični gradient energije po vseh koordinatah (centralne diference). Vrne array (n, 2)."""
    g = np.zeros_like(t)
    # TODO: za vsako oglišče i in koordinato d: (E(t + h e_id) - E(t - h e_id)) / (2h)
    pass
    return g


def gradient_ploscine(t):
    """Analitični gradient ploščine: dA/dx_i = (y_{i+1} - y_{i-1})/2, dA/dy_i = (x_{i-1} - x_{i+1})/2."""
    # TODO: np.roll
    pass


def gradientni_spust(t, ref, eta=0.05, n_iter=2000, tol=1e-10):
    """Relaksacija celice z gradientnim spustom. Konča, ko je E < tol.
    Vrne (t, zgodovina E, zgodovina |grad E|)."""
    t = t.copy()
    zE, zg = [energija(t, ref)], []
    for _ in range(n_iter):
        # TODO: izračunaj gradient, premakni oglišča, shrani E in normo gradienta
        break
        if zE[-1] < tol:
            break
    return t, np.array(zE), np.array(zg)


# --- Tkivo: skupni seznam točk, celice so seznami indeksov točk (v pozitivni smeri) ---
def referenca(A0, n):
    """Referenčne vrednosti (A0, P0, l0) pravilnega n-kotnika s ploščino A0."""
    Rr = np.sqrt(2 * A0 / (n * np.sin(2 * np.pi / n)))
    l0 = 2 * Rr * np.sin(np.pi / n)
    return A0, n * l0, l0


def energija_tkiva(tocke, celice, A0):
    """Vsota energij vseh celic; celica k ima referenčno ploščino A0[k]."""
    # TODO: vsota energija(tocke[c], referenca(A0[k], len(c))) po celicah
    pass


def gradient_tkiva(tocke, celice, A0, h=1e-6):
    """Numerični gradient energije tkiva po vseh točkah. Točka vpliva le na celice, ki jo vsebujejo."""
    g = np.zeros_like(tocke)
    for i in range(len(tocke)):
        moje = [k for k, c in enumerate(celice) if i in c]
        for d in range(2):
            tp, tm = tocke.copy(), tocke.copy()
            tp[i, d] += h
            tm[i, d] -= h
            g[i, d] = sum(energija(tp[celice[k]], referenca(A0[k], len(celice[k])))
                          - energija(tm[celice[k]], referenca(A0[k], len(celice[k]))) for k in moje) / (2 * h)
    return g


def relaksacija_tkiva(tocke, celice, A0, eta=0.05, n_iter=1000, tol=1e-10):
    """Gradientni spust za vse točke tkiva. Skupne točke se premikajo hkrati za vse celice."""
    tocke = tocke.copy()
    zE = [energija_tkiva(tocke, celice, A0)]
    for _ in range(n_iter):
        # TODO: tocke -= eta * gradient; shrani energijo
        break
        if zE[-1] < tol:
            break
    return tocke, np.array(zE)


def rast(tocke, celice, A0, rastoce, faktor=2.0, koraki=10, eta=0.05):
    """Referenčno ploščino celic iz seznama rastoce v korakih poveča za faktor; po vsakem koraku
    relaksira tkivo. Vrne (tocke, nov A0, ploščine rastočih celic po korakih)."""
    A0 = list(A0)
    zacetne = {k: A0[k] for k in rastoce}
    zg = [[ploscina(tocke[celice[k]]) for k in rastoce]]
    for s in range(1, koraki + 1):
        # TODO: A0[k] = (1 + (faktor - 1) s / koraki) * začetna; relaksacija (300 iteracij)
        pass
    return tocke, A0, np.array(zg)


def vstavi_v_sosede(celice, a, b, nova, razen):
    """Točko nova vstavi med točki a in b v vseh celicah (razen celice razen), ki imajo stranico (a, b)."""
    for k, c in enumerate(celice):
        if k == razen:
            continue
        for i in range(len(c)):
            if {c[i], c[(i + 1) % len(c)]} == {a, b}:
                c.insert(i + 1, nova)      # pri paru (zadnja, prva) se točka doda na konec
                break


def deli_celico(tocke, celice, k, kot):
    """Celico k razdeli s premico skozi njeno težišče s smerjo kot (radiani).
    Na obeh presekanih stranicah ustvari novo točko in jo vstavi tudi v sosednjo celico.
    Celico k nadomesti prva hči, druga hči se doda na konec seznama celic.
    Hčeri si delita stranico med novima točkama. Vrne (tocke, celice, (m1, m2))."""
    c = list(celice[k])
    P = tocke[c]
    nor = np.array([-np.sin(kot), np.cos(kot)])      # normala premice
    s = (P - tezisce(P)) @ nor                       # predznačena razdalja oglišč od premice
    presek = []                                      # (položaj stranice v celici, indeks nove točke)
    for i in range(len(c)):
        j = (i + 1) % len(c)
        if s[i] * s[j] < 0:                          # stranica (c[i], c[j]) seka premico
            # TODO: presečišče p, dodaj ga v tocke, zapomni si indeks in ga vstavi v sosednje celice
            pass
    (i1, m1), (i2, m2) = presek
    # TODO: hči 1 = [m1, oglišča od i1+1 do i2, m2], hči 2 = [m2, oglišča od i2+1 do konca in od začetka do i1, m1]
    pass
    return tocke, celice, (m1, m2)


def narisi_tkivo(ax, tocke, celice, oznake=False):
    barve = plt.cm.tab10(np.arange(len(celice)) % 10)
    for c, b in zip(celice, barve):
        narisi_celico(ax, tocke[c], b)
    if oznake:
        for i, p in enumerate(tocke):
            ax.annotate(str(i), p, fontsize=7, xytext=(3, 3), textcoords="offset points")


def dolga_os(t):
    """Smer (kot) dolge osi celice iz lastnih vektorjev kovariančne matrike oglišč."""
    w, v = np.linalg.eigh(np.cov((t - tezisce(t)).T))
    return np.arctan2(v[1, -1], v[0, -1])

# Testno območje

In [ ]:
ref = referencne(R, n)
t0 = poligon(R, n, eps=0.3, rng=np.random.default_rng(7))

# Analitični in numerični gradient ploščine (K = 1, ostali uteži 0 -> dE/dr = (A - A0) dA/dr)
g_num = np.zeros_like(t0)
for i in range(n):
    for d in range(2):
        tp, tm = t0.copy(), t0.copy()
        tp[i, d] += 1e-6
        tm[i, d] -= 1e-6
        g_num[i, d] = (ploscina(tp) - ploscina(tm)) / 2e-6
print("max |dA analitično - numerično| =", np.abs(gradient_ploscine(t0) - g_num).max())

# P4: delitev kvadrata s premico y = 0
tk, ck, _ = deli_celico(np.array([[1, 1], [-1, 1], [-1, -1], [1, -1]], dtype=float), [[0, 1, 2, 3]], 0, 0.0)
print("hčeri kvadrata:", ck, [ploscina(tk[c]) for c in ck], "skupni točki:", sorted(set(ck[0]) & set(ck[1])))

# Naloga 1 - Slika 2

In [ ]:
k1 = 10.0
etas = [0.01, 0.04, 0.06, 0.09, 0.11]
fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
# TODO: x_n in E_n = k (x_n - 1)^2 za vse eta (x začetni = 2); Preglednica 1
fig.tight_layout()
plt.show()

# Naloga 2

In [ ]:
# TODO: gradientni spust za popačen šestkotnik pri več eta; število izračunov energije (4n na iteracijo); Preglednica 2

# Naloga 2 - Slika 3

In [ ]:
t_r, E_h, g_h = gradientni_spust(t0, ref, eta=0.1)
fig = plt.figure(figsize=(12, 3.5))
ax = fig.add_subplot(131)
# TODO: začetna in relaksirana celica, E in |grad E| po iteracijah
fig.tight_layout()
plt.show()

# Naloga 3 - Slika 4

In [ ]:
# Tkivo z eno celico: pravilni šestkotnik
tocke = poligon(R, n)
celice = [list(range(n))]
A0 = [ref[0]]
# TODO: rast do dvojne ploščine v 10 korakih; ploščina po korakih (Preglednica 3)
tocke_vel, A0_vel, A_zg = tocke, A0, np.array([[ploscina(tocke)]])
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121)
narisi_celico(ax, tocke, "tab:blue")
narisi_celico(ax, tocke_vel, "tab:orange")
ax = fig.add_subplot(122)
ax.plot(A_zg[:, 0] / A_zg[0, 0], "o-")
ax.set_xlabel("korak rasti")
ax.set_ylabel("$A/A_0$")
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 5

In [ ]:
rng_d = np.random.default_rng(3)
kot = rng_d.uniform(0, np.pi)                    # naključna smer premice skozi težišče
celice1 = [list(c) for c in celice]
# TODO: delitev zrasle celice; skupne točke hčera; relaksacija obeh hčera k A0 (Preglednica 4)
tocke1 = tocke_vel
fig = plt.figure(figsize=(12, 4))
for k, (T_, C_, naslov) in enumerate(((tocke_vel, celice, "mati"), (tocke1, celice1, "po delitvi"),
                                      (tocke1_r, celice1, "po relaksaciji"))):
    ax = fig.add_subplot(1, 3, k + 1)
    narisi_tkivo(ax, T_, C_, oznake=True)
    ax.set_title(naslov, loc="left")
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 6

In [ ]:
# TODO: druga generacija: obe hčeri zrasteta, se razdelita (naključni koti) in tkivo se relaksira
tocke2, celice2 = tocke1_r, celice1
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(121)
narisi_tkivo(ax, tocke2, celice2)
ax.set_title("po drugi delitvi", loc="left")
ax = fig.add_subplot(122)
narisi_tkivo(ax, tocke2_r, celice2, oznake=True)
ax.set_title("po relaksaciji", loc="left")
fig.tight_layout()
plt.show()